# Lecture 6 · Notebook 3 — Test-time compute: sample, vote, verify, and "Wait"
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture06/lecture06_3_test_time_compute.ipynb)

**Companion to [Chapters 6–7 of the video lecture](https://nlp-course-cuhksz.github.io/lecture06.html#p7-s122)** (slides 96–98 and 122–123). Reasoning models get better when they
spend more compute at inference: think longer, or think in parallel and pick an answer. Here you measure this yourself with a small open
model, **Qwen2.5-0.5B-Instruct**, on grade-school math word problems (**GSM8K**), where the final answer can be checked automatically:

1. solve problems step by step and check the answers with a rule (the RLVR-style verifier),
2. sample $k$ solutions per problem and compare **pass@k** (an oracle verifier picks a right one if there is one) with **majority
   vote** (no verifier at all), as a function of $k$ and of the tokens spent (slides 98 and 122–123),
3. try s1's **budget forcing**: when the model stops, append "Wait" and let it think again (slide 98).

**Use a GPU:** *Runtime → Change runtime type → T4 GPU*; the full run (40 problems × 8 samples) takes about 5–10 minutes. On a CPU the
notebook automatically uses a much smaller setting (6 problems × 4 samples), which still takes roughly 20–30 minutes, so prefer the GPU. The model download is about 1 GB.

In [ ]:
!pip -q install -U transformers accelerate

In [ ]:
import re, json, math, time, collections, requests
import numpy as np, torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
N_QUESTIONS, K, MAX_NEW_TOKENS = (40, 8, 512) if device == "cuda" else (6, 4, 384)
tok = AutoTokenizer.from_pretrained(MODEL_ID, padding_side="left")
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16 if device == "cuda" else torch.float32).to(device).eval()
print(device, "| problems:", N_QUESTIONS, "| samples per problem:", K)

## 1 · Data and a rule-based verifier
GSM8K test problems come with a worked solution that ends in `#### <number>`. We ask the model to put its final answer in `\boxed{}`
(the format Qwen's math models are trained to use) and compare numbers. This is exactly the kind of checkable reward that RLVR trains
on (slides 69–70).

In [ ]:
URL = "https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data/test.jsonl"
rows = [json.loads(l) for l in requests.get(URL, timeout=60).text.splitlines()[:N_QUESTIONS]]
questions = [r["question"] for r in rows]
gold = [r["answer"].split("####")[-1].strip().replace(",", "") for r in rows]

def extract(text):
    boxed = re.findall(r"\\boxed\{([^{}]*)\}", text)
    cand = boxed[-1] if boxed else (re.findall(r"-?\d[\d,]*\.?\d*", text) or [""])[-1]
    cand = cand.replace(",", "").replace("$", "").strip()
    m = re.search(r"-?\d+\.?\d*", cand)
    return m.group(0).rstrip(".") if m else None

def is_correct(pred, g):
    try: return pred is not None and abs(float(pred) - float(g)) < 1e-6
    except ValueError: return False

print(questions[0]); print("gold answer:", gold[0])

In [ ]:
SYSTEM = "Please reason step by step, and put your final answer within \\boxed{}."
def chat(q):
    return tok.apply_chat_template([{"role": "system", "content": SYSTEM}, {"role": "user", "content": q}],
                                   tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def generate(prompts, n=1, greedy=False, max_new_tokens=MAX_NEW_TOKENS, batch=8):
    # returns, for each prompt, a list of n (text, number of generated tokens)
    out = []
    for i in range(0, len(prompts), batch):
        enc = tok(prompts[i:i + batch], return_tensors="pt", padding=True).to(device)
        kw = dict(do_sample=False) if greedy else dict(do_sample=True, temperature=0.7, top_p=0.95)
        g = model.generate(**enc, max_new_tokens=max_new_tokens, num_return_sequences=n, pad_token_id=tok.eos_token_id, **kw)
        g = g[:, enc["input_ids"].shape[1]:]
        texts = tok.batch_decode(g, skip_special_tokens=True)
        lens = (g != tok.pad_token_id).sum(1).tolist()
        for j in range(0, len(texts), n): out.append(list(zip(texts[j:j + n], lens[j:j + n])))
    return out

t0 = time.time()
greedy = [r[0] for r in generate([chat(q) for q in questions], greedy=True)]
greedy_ok = [is_correct(extract(t), g) for (t, _), g in zip(greedy, gold)]
print(f"greedy accuracy: {np.mean(greedy_ok):.1%}  ({time.time() - t0:.0f}s, {np.mean([l for _, l in greedy]):.0f} tokens per answer)")
print("\n--- one solution ---\n" + greedy[0][0][:800])

## 2 · Think in parallel: pass@k versus majority vote
Now sample $K$ solutions per problem at temperature 0.7.

* **pass@k:** is at least one of $k$ samples right? That is what you would get with a **perfect verifier** that picks a correct sample
  whenever there is one (best-of-N with a verifier, slide 123). We use the unbiased estimator from the Codex paper.
* **maj@k:** take the most common final answer among $k$ samples (self-consistency). No verifier needed.

In [ ]:
t0 = time.time()
samples = generate([chat(q) for q in questions], n=K, batch=max(1, 32 // K))
print(f"sampled {N_QUESTIONS * K} solutions in {time.time() - t0:.0f}s")
preds = [[extract(t) for t, _ in s] for s in samples]
correct = np.array([[is_correct(p, g) for p in ps] for ps, g in zip(preds, gold)])
tokens = np.array([[l for _, l in s] for s in samples])

def pass_at_k(n, c, k): return 1.0 if n - c < k else 1.0 - math.comb(n - c, k) / math.comb(n, k)
def maj_at_k(ps, g, k):
    votes = collections.Counter(p for p in ps[:k] if p is not None)
    return bool(votes) and is_correct(votes.most_common(1)[0][0], g)

ks = [k for k in (1, 2, 4, 8, 16) if k <= K]
pass_k = [np.mean([pass_at_k(K, c, k) for c in correct.sum(1)]) for k in ks]
maj_k = [np.mean([maj_at_k(ps, g, k) for ps, g in zip(preds, gold)]) for k in ks]
for k, a, b in zip(ks, pass_k, maj_k):
    print(f"k={k:2d}  pass@k {a:.1%}   maj@k {b:.1%}   tokens per problem ≈ {k * tokens.mean():.0f}")

fig, ax = plt.subplots(figsize=(6, 3.8))
ax.plot([k * tokens.mean() for k in ks], pass_k, "o-", label="pass@k (perfect verifier)")
ax.plot([k * tokens.mean() for k in ks], maj_k, "s-", label="maj@k (majority vote)")
ax.axhline(np.mean(greedy_ok), ls=":", c="gray", label="greedy, 1 sample")
ax.set_xscale("log"); ax.set_xlabel("generated tokens per problem (test-time compute)"); ax.set_ylabel("accuracy")
ax.legend(); ax.set_title(f"{MODEL_ID.split('/')[-1]} on {N_QUESTIONS} GSM8K problems"); plt.show()

Two lessons from the lecture show up in one plot:

* **Test-time scaling** (slide 96): spending more tokens per problem buys accuracy, with no change to the weights.
* **The gap between pass@k and maj@k** is what a good verifier is worth (slides 122–123): the model can often produce a right answer,
  but it cannot tell which one it is. Learned verifiers try to close this gap; RL training tries to move pass@k's mass into pass@1
  (Notebook 2).

Majority vote only helps when the right answer is the *most common* one. On problems where the model is usually wrong, voting locks in
the wrong answer, while pass@k still credits the one lucky sample.

## 3 · Think longer: budget forcing with "Wait"
s1 (slide 98) forces longer thinking with a trick: when the model wants to stop, cut off the ending and append **"Wait"**, so it re-reads
its work. We try it on problems the greedy answer got wrong.

In [ ]:
wrong = [i for i, ok in enumerate(greedy_ok) if not ok][:4]
fixed = 0
for i in wrong:
    first = greedy[i][0]
    cut = first.rsplit("\\boxed", 1)[0]                       # drop the final answer, keep the reasoning
    prompt2 = chat(questions[i]) + cut + "\n\nWait, let me double-check this step by step."
    second = generate([prompt2], greedy=True, max_new_tokens=MAX_NEW_TOKENS)[0][0][0]
    p1, p2 = extract(first), extract(second)
    fixed += is_correct(p2, gold[i])
    print(f"Q{i}: gold {gold[i]:>6} | first answer {str(p1):>8} | after 'Wait' {str(p2):>8}")
print(f"\nfixed {fixed} of {len(wrong)} wrong answers")

Sometimes "Wait" fixes an answer, sometimes it does nothing, and sometimes a small model talks itself out of a right answer. That
matches slide 99's warning: a weak model can copy the *words* of reflection ("wait", "let me check") without the skill behind them.
Real reasoning models learn *when* reflection pays off through RL (Chapter 8: DeepSeek-R1 and SimpleRL).

## Exercises
1. Change the temperature in `generate` to 0.3 and 1.0. How do pass@k and maj@k change? Why does diversity help pass@k more than maj@k?
2. Build a weak verifier with no labels: score each sample by its average token log-probability and pick the best one. Is it closer to
   maj@k or to pass@k?
3. Swap in `Qwen/Qwen2.5-Math-1.5B-Instruct` (GPU only, about 3 GB). How much of the gap between greedy and pass@8 is closed by a model
   that was trained with math-specific SFT and RL?

**Previous:** [Notebook 2 — GRPO and pass@k on a verifiable task]({colab(N2)}) · **Back to the [lecture page]({SITE})**